## Read in data 

In [1]:
import io
import pandas as pd

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV))

In [2]:
stream

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,-999.0,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


Before writing any more code, agree with your team on the answers to these three questions and write them in a markdown cell.

How many monitoring sites are in this dataset?
> There are three moonitoring sites.

What is a plausible range for stream water temperature in degrees Celsius?
> A plausible range for stream water temp is 10-25 degrees Celcius.

Roughly what mean water temperature do you expect across all measurements?
> The mean water temperature is 16.

In [3]:
stream['water_temp_c'].mean()

-153.8111111111111

## A worked example

In [ ]:
# Step 1: read the error. Running the cell gives
stream['discharge_cfs'] * 0.0283168

TypeError: can't multiply sequence by non-int of type 'float'

In [7]:
# Step 2: look at the data
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l      object
qc_flag           object
dtype: object

In [8]:
# Step 3: isolate
stream['discharge_cfs'].iloc[0] * 0.0283168   # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

TypeError: can't multiply sequence by non-int of type 'float'

In [9]:
# Step 4: form a hypothesis and test it
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [10]:
# Step 5: fix and verify -- Remove the separator, convert, then check the result against something you know
discharge_cms = (stream['discharge_cfs']
                    .str.replace(',', '', regex=False)
                    .astype(float)) * 0.0283168

discharge_cms.head(3)


0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [11]:
# tep 6: Create a guard
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

And if none of this had worked? Suppose the protocol had run out of road and you still did not know why the multiplication failed. At that point you would ask someone, and you would ask by building a minimal reproducible example, which is a two-row version of the problem that another person can run in isolation.



In [12]:
import pandas as pd
df = pd.DataFrame({'discharge_cfs': ['1,240', '980']})
df['discharge_cfs'] * 0.0283168     # TypeError: can't multiply sequence by non-int

TypeError: can't multiply sequence by non-int of type 'float'

The Error Log 
| What I expected | What Happened |  My hypothesis |  What I changed  |  Result  |
| --------------- | ------------- | -------------- | ---------------- | -------- |
|   Discharge in m³/s, around 35  |    TypeError: can't multiply sequence by non-int  |  discharge_cfs is text because of thousands separators  |  .str.replace(',', '') then .astype(float)  |  Works, 35.11 m³/s, guard added  |
|       strip Site_id   |   Series' object has no attribute 'strip'  |   Our hypothesis is that .str needs to precede .strip() for Series objects  |  We added .str before .strip()  |  Works, no error returned and column removed white space |
|  Change data type to float |  TypeError: could not convert string to float: '<0.05'  |  Our hypothesis is that because of the negative values, there can be no conversion |  We took care of negative values  |  Works, no error and data type changed  |

## On your own

In [27]:
# a. 
# stream['site_id'].strip()

# Step 1: Read the error -> The issue is that strip can not be used by itself 



# Step 2: Look at the data, returns object
print(stream['site_id'].dtypes)


# Step 3: isolate
print(stream['site_id'].iloc[0].strip())
print(type(stream['site_id'].iloc[0]))

# After isolation, the first value in site_id was returned. Using strip() after .iloc[] turned the value into a string. 
# The smallest segment of the code that still cause the error is the original `stream['site_id].strip()`


# Step 4: form a hypothesis  -> Our hypothesis is that .str needs to precede .strip() for Series objects


# Step 5:Fix and verify 
stream['site_id'].str.strip()


# Step 6:

object
SB-01
<class 'str'>


0     SB-01
1     SB-01
2     SB-01
3     SB-01
4     sb-01
5     SB-01
6     SB-02
7     SB-02
8     SB-02
9     SB-02
10    SB-02
11    sb-02
12    SB-03
13    SB-03
14    SB-03
15    SB-03
16    SB-03
17    SB-03
Name: site_id, dtype: object

## 4. Bugs that do not raise errors


In [29]:
# a. Run the following cell and compare the result against your expectation from 1c-iii.
stream['water_temp_c'].mean()

-153.8111111111111

i. The cell runs without error. Is the answer right, and how do you know?

    No, the answer is incorrect because a mean value of -153 degress C for stream water is not reasonable.

ii. Explain in one sentence why the result is what it is.

    The result is -153 degrees C because of the -999 values that populate these observations are skewing the results.


iii. Fix it. Several approaches work, and the best one handles the problem at the moment the data is read, so that no later calculation can ever touch the sentinel.

    We can assign -999 values to NA when we initially read in the data.



In [32]:
stream = pd.read_csv(io.StringIO(CSV), na_values = -999.99)

stream

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,NaN,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


In [ ]:
# b. Run the following cell and compare the result against your expectation from 1c-i. 

stream['site_id'].nunique()

7